In [1]:
import numpy as np
from numpy import cos,pi,sqrt,log10
from astropy.io import ascii
from astropy.coordinates import SkyCoord
from astropy.io import fits
from gammapy.irf import EffectiveAreaTable2D
import ctadata
import os
import astropy.units as u

In [2]:
data_folder = "./LST_data/"
DL3_folder = "/pnfs/cta.cscs.ch/lst/DL3/"
run_catalog=ascii.read(data_folder + 'LST_source_catalog.ecsv')

#CAREFUL ENERGY BINS OF CRAB PSF MUST MATCH THESE ONES (see UL calculation)
e_bins=np.logspace(np.log10(0.01), np.log10(10), 11)
e_mins=e_bins[:-1]
e_maxs=e_bins[1:]
e=np.sqrt(e_mins*e_maxs)
de=e_maxs-e_mins

th2_bins=np.linspace(0,0.16,34)
th2=(th2_bins[1:]+th2_bins[:-1])/2.

In [3]:
def find_and_process_files(runlist, gheff, 
                          ra_obj, dec_obj, bkg_subtraction_radius,
                          e_mins, e_maxs, e, th2_bins,
                          cts_s, cts_b, t_expo):
    pointing_sep_avg = 0
    pointing_sep_counter = 0
    counter = 0
    n_energy_bins = len(e)
    n_theta_bins = len(th2_bins) - 1
    
    cts_effarea_corr = np.zeros(
        (n_energy_bins, n_theta_bins),
        dtype=float
    )
    
    cts_effarea_corr_err = np.zeros(
        (n_energy_bins, n_theta_bins),
        dtype=float
    )
    
    for ind in range(len(runlist)):
        r = runlist[ind]
        for i in range(len(run_catalog)):
            run = run_catalog[i]['Run ID']
            if run == r:
                date = run_catalog[i]['Date directory'].replace('-', '')
                vers = ctadata.list_dir(DL3_folder + date)
                for ver in vers:
                    if ver[0] == 'v':
                        tailcuts = ctadata.list_dir(DL3_folder + date + '/' + ver)
                        for tailcut in tailcuts:
                            nsbs = ctadata.list_dir(DL3_folder + date + '/' + ver + '/' + tailcut)
                            for nsb in nsbs:
                                versions = ctadata.list_dir(DL3_folder + date + '/' + ver + '/' + tailcut + '/' + nsb)
                                for version in versions:
                                    tags = ctadata.list_dir(DL3_folder + date + '/' + ver + '/' + tailcut + '/' + nsb + '/' + version + '/std')
                                    for tag in tags:
                                        src_dependences = ctadata.list_dir(DL3_folder + date + '/' + ver + '/' + tailcut + '/' + nsb + '/' + version + '/std/' + tag)
                                        for src_dep in src_dependences:
                                            point_or_full = ctadata.list_dir(DL3_folder + date + '/' + ver + '/' + tailcut + '/' + nsb + '/' + version + '/std/' + tag + '/' + src_dep)
                                            for p_f in point_or_full:
                                                wobbles = ctadata.list_dir(DL3_folder + date + '/' + ver + '/' + tailcut + '/' + nsb + '/' + version + '/std/' + tag + '/' + src_dep + '/' + p_f)
                                                for wob in wobbles:
                                                    gheffs = ctadata.list_dir(DL3_folder + date + '/' + ver + '/' + tailcut + '/' + nsb + '/' + version + '/std/' + tag + '/' + src_dep + '/' + p_f + '/' + wob)
                                                    for gh in gheffs:
                                                        if gheff in gh:
                                                            irfs = ctadata.list_dir(DL3_folder + date + '/' + ver + '/' + tailcut + '/' + nsb + '/' + version + '/std/' + tag + '/' + src_dep + '/' + p_f + '/' + wob + '/' + gh)
                                                            for irf in irfs:
                                                                files = ctadata.list_dir(DL3_folder + date + '/' + ver + '/' + tailcut + '/' + nsb + '/' + version + '/std/' + tag + '/' + src_dep + '/' + p_f + '/' + wob + '/' + gh + '/' + irf)
                                                                
                                                                if run < 10000:
                                                                    fname = 'dl3_LST-1.Run0' + str(run) + '.fits'
                                                                else:
                                                                    fname = 'dl3_LST-1.Run' + str(run) + '.fits'
                                                                
                                                                if fname in files:
                                                                    file_path = DL3_folder + date + '/' + ver + '/' + tailcut + '/' + nsb + '/' + version + '/std/' + tag + '/' + src_dep + '/' + p_f + '/' + wob + '/' + gh + '/' + irf + '/' + fname
                                                                    ctadata.fetch_and_save_file_or_dir(file_path)
                                                                    effarea_hdu = EffectiveAreaTable2D.read(fname, hdu="EFFECTIVE AREA")
                                                                    
                                                                    with fits.open(fname) as hdul:
                                                                        header = hdul[1].header
                                                                        dat = header['DATE-OBS']
                                                                        t_expo += header['LIVETIME']
                                                                        print(ind, date, fname, header['LIVETIME'])
                                                                        
                                                                        ra_pnt = header['RA_PNT']
                                                                        dec_pnt = header['DEC_PNT']
                                                                        dra = ra_obj - ra_pnt
                                                                        ddec = dec_obj - dec_pnt
                                                                        
                                                                        ra_bkg = ra_pnt - dra
                                                                        dec_bkg = dec_pnt - ddec
                                                                        coords_bkg = SkyCoord(ra_bkg, dec_bkg, unit='degree')
                                                                        coords_obj = SkyCoord(ra_obj, dec_obj, unit='degree')

                                                                        sep_pnt = SkyCoord(ra_pnt, dec_pnt, unit='degree').separation(coords_obj).deg
                                                                        pointing_sep_counter += 1
                                                                        pointing_sep_avg += sep_pnt
                                                                        
                                                                        events = hdul['EVENTS'].data
                                                                        coords = SkyCoord(events['RA'], events['DEC'], unit='degree')
                                                                        
                                                                        seps = coords.separation(coords_obj).deg
                                                                        seps_b = coords.separation(coords_bkg).deg
                                                                        energies = events['ENERGY']
                    
                                                                        cts_effarea_corr_tmp = np.zeros_like(cts_effarea_corr)
                                                                        cts_effarea_corr_err_tmp = np.zeros_like(cts_effarea_corr_err)
                                                                        for i in range(len(e)):
                                                                            m_s = (energies > e_mins[i]) * (energies < e_maxs[i]) * (seps < bkg_subtraction_radius)
                                                                            h_s = np.histogram(seps[m_s]**2, bins=th2_bins)
                                                                            
                                                                            m_b = (energies > e_mins[i]) * (energies < e_maxs[i]) * (seps_b < bkg_subtraction_radius)
                                                                            h_b = np.histogram(seps_b[m_b]**2, bins=th2_bins)
                                                                            
                                                                            cts_s[i] += h_s[0]
                                                                            cts_b[i] += h_b[0]
                                                                    
                                                                            # If using this remember to square the error after the fuction.
                                                                            aeff = effarea_hdu.evaluate(energy_true=e[i] * u.TeV, offset=bkg_subtraction_radius * u.deg).to(u.cm**2).value
                                                                            cts_effarea_corr_tmp[i] += (h_s[0] - h_b[0]) / aeff
                                                                            cts_effarea_corr_err_tmp[i] += (h_s[0] + h_b[0]) / aeff**2

                                                                        cts_effarea_corr += cts_effarea_corr_tmp
                                                                        cts_effarea_corr_err += cts_effarea_corr_err_tmp
                                                                        
                                                                    os.remove(fname)
                                                                    counter += 1
                                                                    if counter >= countermax:
                                                                        cts_effarea_corr_err = np.sqrt(cts_effarea_corr_err)
                                                                        pointing_sep_avg /= pointing_sep_counter
    
                                                                        return cts_s, cts_b, t_expo, pointing_sep_avg, cts_effarea_corr, cts_effarea_corr_err
    cts_effarea_corr_err = np.sqrt(cts_effarea_corr_err)
    pointing_sep_avg /= pointing_sep_counter
    
    return cts_s, cts_b, t_expo, pointing_sep_avg, cts_effarea_corr, cts_effarea_corr_err

In [4]:
name = "Mrk501"
runlist = np.load(data_folder + 'good_runs_' + name + '_Zd_30.0.npy')
gheff='gheff0.9'
Zdcut=30

coords_obj=SkyCoord.from_name(name)
ra_obj=coords_obj.icrs.ra.deg
dec_obj=coords_obj.icrs.dec.deg
ra_obj,dec_obj
cdec=cos(dec_obj*pi/180.)

In [5]:
countermax = 200
bkg_subtraction_radius = 0.4

cts_s = np.zeros((len(e), len(th2)))
cts_b = np.zeros((len(e), len(th2)))

t_expo = 0

cts_s, cts_b, t_expo, pointing_seps_avg, cts_effarea_corr, cts_effarea_corr_err = find_and_process_files(
    runlist=runlist,
    gheff=gheff,
    ra_obj=ra_obj,
    dec_obj=dec_obj,
    bkg_subtraction_radius=bkg_subtraction_radius,
    e_mins=e_mins,
    e_maxs=e_maxs,
    e=e,
    th2_bins=th2_bins,
    cts_s=cts_s,
    cts_b=cts_b,
    t_expo=t_expo,
)
print(f"Total exposure: {t_expo/3600}h")

0 20210317 dl3_LST-1.Run04139.fits 1130.0273713094493
1 20210317 dl3_LST-1.Run04140.fits 1155.6040742903724
2 20210320 dl3_LST-1.Run04191.fits 1090.8443539007542
3 20210320 dl3_LST-1.Run04192.fits 1057.6035122864566
4 20210320 dl3_LST-1.Run04193.fits 1011.8741674186106
5 20210320 dl3_LST-1.Run04194.fits 468.0308600181666
7 20210321 dl3_LST-1.Run04212.fits 851.6007298327497
8 20210321 dl3_LST-1.Run04213.fits 853.1299317712834
9 20210321 dl3_LST-1.Run04214.fits 845.836385717532
10 20210321 dl3_LST-1.Run04215.fits 852.2641518893802
11 20210417 dl3_LST-1.Run04473.fits 572.5225141427318
12 20210417 dl3_LST-1.Run04474.fits 1132.3215512024444
13 20210417 dl3_LST-1.Run04475.fits 1137.6344790185653
14 20210417 dl3_LST-1.Run04476.fits 571.5857753250073
15 20210418 dl3_LST-1.Run04483.fits 859.8120407428157
16 20210418 dl3_LST-1.Run04484.fits 1138.5015524738958
17 20210419 dl3_LST-1.Run04496.fits 1021.9748575572272
18 20210419 dl3_LST-1.Run04497.fits 1022.8591745170011
19 20210419 dl3_LST-1.Run044

In [6]:
np.savez(
    "LST_counts.npz",
    e=e,
    cts_s=cts_s,
    cts_b=cts_b,
    t_expo=t_expo,
    pointing_seps_avg=pointing_seps_avg,
    cts_effarea_corr=cts_effarea_corr,
    cts_effarea_corr_err=cts_effarea_corr_err
)